## Analyzing Factors Associated with Traffic Crash Severity in California Using 2025 CCRS Data

**Team Members:**  
- Jonathan Mitchell  
- Daniel Ortega Rubio 
- Sherlyn Ballestero Cruz

**Research Question:**  
Are environmental factors (weather, lighting, road conditions) and temporal factors (time of day, day of the week) significantly associated with crash severity in California 2025 traffic crashes?

This notebook follows the full data science pipeline.


### Step 1: Get Data

Load the dataset, clean column names, select only useful columns and convert the date.


In [1]:
import pandas as pd
import numpy as np
import altair as alt
from altair import Chart, X, Y
from IPython.display import display

# Official dataset URL
url = "https://data.ca.gov/dataset/80c6a49d-c6b3-40ba-86d8-379c9741b4be/resource/9f4fc839-122d-4595-a146-43bc4ed16f46/download/crashes_2025.csv"

# Load the dataset
df = pd.read_csv(url, low_memory=False)

# Clean column names
df.columns = df.columns.str.strip()

# Keep only the columns we need (reduces memory a lot)
useful_cols = [
    'Collision Id', 'Crash Date Time', 'Day Of Week',
    'Weather 1', 'LightingDescription', 'Road Condition 1',
    'NumberInjured', 'NumberKilled', 'Collision Type Description'
]
df = df[useful_cols].copy()

print("Shape after selecting columns:", df.shape)
display(df.head())


Shape after selecting columns: (402541, 9)


,Collision Id,Crash Date Time,Day Of Week,Weather 1,LightingDescription,Road Condition 1,NumberInjured,NumberKilled,Collision Type Description
0,4550265,1/18/2025 3:20:00 PM,Saturday,CLEAR,DAYLIGHT,NO UNUSUAL CONDITIONS,1.0,0.0,OVERTURNED
1,4550264,1/10/2025 8:28:00 AM,Friday,CLEAR,DAYLIGHT,NO UNUSUAL CONDITIONS,1.0,0.0,REAR END
2,4550263,1/23/2025 8:45:00 AM,Thursday,CLEAR,DAYLIGHT,NO UNUSUAL CONDITIONS,0.0,0.0,SIDE SWIPE
3,4550262,1/15/2025 12:30:00 PM,Wednesday,WIND,DAYLIGHT,NO UNUSUAL CONDITIONS,0.0,0.0,HIT OBJECT
4,4550261,1/20/2025 12:30:00 PM,Monday,CLEAR,DAYLIGHT,NO UNUSUAL CONDITIONS,3.0,0.0,BROADSIDE


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 402541 entries, 0 to 402540
Data columns (total 9 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Collision Id                402541 non-null  int64  
 1   Crash Date Time             402541 non-null  str    
 2   Day Of Week                 402541 non-null  str    
 3   Weather 1                   401023 non-null  str    
 4   LightingDescription         401719 non-null  str    
 5   Road Condition 1            402541 non-null  str    
 6   NumberInjured               383533 non-null  float64
 7   NumberKilled                382863 non-null  float64
 8   Collision Type Description  401586 non-null  str    
dtypes: float64(2), int64(1), str(6)
memory usage: 56.8 MB


In [3]:
missing_values = df.isnull().sum()

missing_percentage = (missing_values / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing value:': missing_values,
    'Percentage missing:': missing_percentage.round(2)
})

display(missing_summary)

,Missing value:,Percentage missing:
Collision Id,0,0.00
Crash Date Time,0,0.00
Day Of Week,0,0.00
Weather 1,1518,0.38
LightingDescription,822,0.20
Road Condition 1,0,0.00
NumberInjured,19008,4.72
NumberKilled,19678,4.89
Collision Type Description,955,0.24


In [4]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate collision IDs:", df['Collision Id'].duplicated().sum())

Duplicate rows: 0
Duplicate collision IDs: 0


In [5]:
df['Crash Date Time'] = pd.to_datetime(
    df['Crash Date Time'],
    format='%m/%d/%Y %I:%M:%S %p',
    errors='coerce'
)

df['Hour'] = df['Crash Date Time'].dt.hour

print("Date range:",
      df['Crash Date Time'].min(),
      "to",
      df['Crash Date Time'].max())
print("Missing or invalid crash dates:", df['Crash Date Time'].isnull().sum())

Date range: 2025-01-01 00:00:00 to 2025-12-31 23:59:00
Missing or invalid crash dates: 0


### Step 2: Exploratory Analysis and Visualization

We create a sample for visualization so Altair stays fast. All charts use pre-aggregated data.


In [7]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Chart 1 - Crashes by day of the week
day_counts = (
    df.dropna(subset=['Day Of Week'])
          .groupby('Day Of Week')
          .size()
          .reset_index(name='count')
)

display(
    Chart(day_counts).mark_bar().encode(
        x=X('Day Of Week', sort=day_order, title='Day of Week'),
        y='count'
    ).properties(title='Number of Crashes by Day of Week', width=500)
)

# Chart 2 - Crashes by hour of the day
hour_counts = (
    df.dropna(subset=['Hour'])
          .groupby('Hour')
          .size()
          .reset_index(name='count')
)

display(
    Chart(hour_counts).mark_bar().encode(
        x=X('Hour:O', title='Hour of Day'),
        y='count'
    ).properties(title='Number of Crashes by Hour of Day', width=600)
)

# Chart 3 - Top weather conditions
top_weather = df['Weather 1'].value_counts().head(8).index
weather_counts = (
    df[df['Weather 1'].isin(top_weather)]
          .groupby('Weather 1')
          .size()
          .reset_index(name='count')
)

display(
    Chart(weather_counts).mark_bar().encode(
        x='count',
        y=Y('Weather 1', sort='-x', title='Weather')
    ).properties(title='Top Weather Conditions', width=500)
)

# Chart 4 - Lighting conditions
lighting_counts = (
    df.dropna(subset=['LightingDescription'])
          .groupby('LightingDescription')
          .size()
          .reset_index(name='count')
)

display(
    Chart(lighting_counts).mark_bar().encode(
        x='count',
        y=Y('LightingDescription', sort='-x', title='Lighting')
    ).properties(title='Lighting Conditions', width=500)
)


alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

### Step 3: Data Cleaning

Create the Severity variable (vectorized = fast), clean categories, and drop rows with missing key values.


In [8]:
for col in ['NumberKilled', 'NumberInjured']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

    df[col] = df[col].where((df[col] >= 0) & (df[col] % 1 == 0))

print(df[['NumberKilled', 'NumberInjured']].isnull().sum())

NumberKilled     19680
NumberInjured    19010
dtype: int64


In [9]:
# Create Severity column (vectorized - much faster than apply)
killed = df['NumberKilled']
injured = df['NumberInjured']

df['Severity'] = np.select(
    [
        killed > 0, (killed == 0) & (injured > 0), (killed == 0) & (injured == 0)
    ],
    ['Fatal', 'Injury', 'PDO'], 
    default='UNKNOWN'
)

print("Severity distribution:")
print(df['Severity'].value_counts())
print(df['Severity'].value_counts(normalize=True).round(3))

for col in ['Weather 1', 'LightingDescription', 'Road Condition 1']:
    df[col] = df[col].str.strip().str.upper()
    df[col] = df[col].replace('', np.nan)
    df[col] = df[col].replace('UNK', 'UNKNOWN')
    
# Clean Weather (keep top categories, rest = OTHER)
known_weather = df['Weather 1'].dropna()

top_weather = known_weather[~known_weather.isin(['UNKNOWN', 'OTHER'])].value_counts().nlargest(7).index
df['Weather_Clean'] = df['Weather 1'].where(df['Weather 1'].isin(top_weather), 'OTHER')

df.loc[
    df['Weather 1'].isna() |
    df['Weather 1'].eq('UNKNOWN'),
    'Weather_Clean'
] = 'UNKNOWN'

# Clean Lighting
known_lighting = df['LightingDescription'].dropna()

top_lighting = known_lighting[~known_lighting.isin(['UNKNOWN', 'OTHER'])].value_counts().nlargest(7).index
df['Lighting_Clean'] = df['LightingDescription'].where(df['LightingDescription'].isin(top_lighting), 'OTHER')

df.loc[
    df['LightingDescription'].isna() |
    df['LightingDescription'].eq('UNKNOWN'),
    'Lighting_Clean'
] = 'UNKNOWN'

# Drop rows missing critical variables
df_clean = df[df['Severity'] != 'UNKNOWN'].dropna(subset=['Hour', 'Day Of Week']).copy()

print("Rows before cleaning:", len(df))
print("Rows after cleaning:", len(df_clean))
print("Rows excluded:", len(df) - len(df_clean))

print("\nWeather_Clean categories:")
print(df_clean['Weather_Clean'].value_counts())

print("\nLighting_Clean categories:")
print(df_clean['Lighting_Clean'].value_counts())


Severity distribution:
Severity
PDO        214861
Injury     164798
UNKNOWN     19682
Fatal        3200
Name: count, dtype: int64
Severity
PDO        0.534
Injury     0.409
UNKNOWN    0.049
Fatal      0.008
Name: proportion, dtype: float64
Rows before cleaning: 402541
Rows after cleaning: 382859
Rows excluded: 19682

Weather_Clean categories:
Weather_Clean
CLEAR             329240
CLOUDY             36514
RAINING            12814
UNKNOWN             1540
FOG/VISIBILITY      1504
WIND                 443
SNOWING              412
OTHER                386
OVERCAST               6
Name: count, dtype: int64

Lighting_Clean categories:
Lighting_Clean
DAYLIGHT                               244674
DARK-STREET LIGHTS                      87617
DARK-NO STREET LIGHTS                   34266
DUSK-DAWN                               14695
UNKNOWN                                   814
DARK-STREET LIGHTS NOT FUNCTIONING*       793
Name: count, dtype: int64


### Step 4: Rescaling / Encoding

Most features are categorical. Use One-Hot Encoding before modeling.


In [10]:
# TODO: Separate features (X) and target (y)

# TODO: Encode categorical variables

# TODO: (Optional) Scale numeric features if the model requires it


### Step 5: Training / Test Split


In [11]:

# TODO: Split the data into training and test sets (use stratification)

# TODO: Check the class distribution in the training set and in the test set

### Step 6: Model Building

Example: Random Forest (handles class imbalance with class_weight).


In [12]:
# TODO: Perform statistical test(s) to evaluate association between factors and Severity

# TODO: Build and train a model using the training data

# TODO: (Optional) Try a second approach and compare results


### Step 7: Model Testing


In [13]:
# TODO: Generate predictions on the test set

# TODO: Calculate and print performance metrics

# TODO: Create and display the confusion matrix

# TODO: Examine which features contribute most to the predictions (if applicable)

### Step 8: Polishing and Presenting

### Conclusions (complete after running the analysis)

**Main findings:**
- TODO

**Answer to the research question:**
- TODO

**Limitations:**
- TODO

**Recommendations:**
- TODO
